# 建立 TensorRT 優化模型

將 PyTorch 訓練好的 `best_steering_model_xy.pth` 轉換為 **TensorRT** 加速模型，
用於 JetBot 上的即時推論。

## 流程
1. 載入 PyTorch 模型（ResNet-18 → 輸出 X, Y）
2. 使用 `torch2trt` 轉換為 TensorRT 引擎
3. 儲存為 `best_steering_model_xy_trt.pth`

## 1. 匯入套件

In [1]:
import torch
import torchvision.models as models
from torch2trt import torch2trt

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

ModuleNotFoundError: No module named 'torch2trt'

## 2. 載入 PyTorch 模型

In [ ]:
PYTORCH_MODEL_PATH = 'best_steering_model_xy.pth'
TRT_MODEL_PATH = 'best_steering_model_xy_trt.pth'

# 建立 ResNet-18 架構（與訓練時相同）
model = models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(512, 2)  # 輸出 X, Y

# 載入訓練好的權重
model.load_state_dict(torch.load(PYTORCH_MODEL_PATH))

# 搬到 GPU 並設為評估模式
device = torch.device('cuda')
model = model.to(device)
model.eval()

print(f'PyTorch 模型已載入: {PYTORCH_MODEL_PATH}')
print(f'模型已搬至 GPU ✓')

## 3. 轉換為 TensorRT 模型

In [ ]:
# 建立範例輸入（batch=1, 3通道, 224x224）
# TensorRT 需要一個範例輸入來建構優化引擎
sample_input = torch.zeros(1, 3, 224, 224).cuda()

print('開始轉換 TensorRT 模型...')
print('（此過程可能需要幾分鐘，請耐心等待）')

# 使用 torch2trt 進行轉換
# fp16_mode=True 啟用半精度加速（Jetson Nano 支援）
model_trt = torch2trt(
    model,
    [sample_input],
    fp16_mode=True
)

print('TensorRT 轉換完成 ✓')

## 4. 驗證 TensorRT 模型

In [ ]:
# 用範例輸入比較 PyTorch 和 TensorRT 的輸出
import numpy as np

test_input = torch.randn(1, 3, 224, 224).cuda()

with torch.no_grad():
    pytorch_output = model(test_input)
    trt_output = model_trt(test_input)

print(f'PyTorch  輸出: {pytorch_output.cpu().numpy()}')
print(f'TensorRT 輸出: {trt_output.cpu().numpy()}')
print(f'最大誤差: {torch.max(torch.abs(pytorch_output - trt_output)).item():.6f}')
print(f'\n驗證通過 ✓' if torch.max(torch.abs(pytorch_output - trt_output)).item() < 0.01 else '\n警告：誤差較大！')

## 5. 儲存 TensorRT 模型

In [ ]:
# 儲存 TensorRT 優化後的模型
torch.save(model_trt.state_dict(), TRT_MODEL_PATH)

print(f'TensorRT 模型已儲存: {TRT_MODEL_PATH}')
print(f'\n下一步：開啟 live_demo_trt.ipynb 進行即時道路跟隨')